Take each zarr file produced by racmo_2km_downscaled/download_upload_to_scratch.ipynb, rechunk to 1 in he time dimension and append to one large zarr for each 

In [1]:
from dask.distributed import Client
client = Client()
client.cluster.scale(5)
client

/srv/conda/envs/notebook/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41007 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jkingslake/load%20NCs/proxy/41007/status,
Dashboard: /user/jkingslake/load%20NCs/proxy/41007/status,Workers: 4
Total threads: 4,Total memory: 14.54 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38323,Workers: 4
Dashboard: /user/jkingslake/load%20NCs/proxy/41007/status,Total threads: 4
Started: Just now,Total memory: 14.54 GiB
Comm: tcp://127.0.0.1:45287,Total threads: 1
Dashboard: /user/jkingslake/load%20NCs/proxy/45281/status,Memory: 3.63 GiB
Nanny: tcp://127.0.0.1:37173,


In [2]:
import xarray as xr
import pandas as pd
from tqdm import tqdm

df = pd.read_csv('record_of_zarrs.csv')

VAR_NAME = "precip"  # change this
DATASET  = "precip"      # change this

df_subset = df[df["dataset"] == DATASET]
f1_sorted = df_subset['zarr_path'].to_list()

OUT_PATH = f"s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/{VAR_NAME}_all_06.zarr"

# Write first file
print(f"Writing first file: {f1_sorted[0]}")
ds = xr.open_zarr(f1_sorted[0], consolidated=True, chunks={})
ds = ds.chunk(time=1)
for var in ds.data_vars:
    if 'chunks' in ds[var].encoding:
        del ds[var].encoding['chunks']
ds.to_zarr(OUT_PATH, mode='w', zarr_format=2)
ds.close()

# Append remaining files
for path in tqdm(f1_sorted[1:], desc="Appending files"):
    ds = xr.open_zarr(path, consolidated=True, chunks={})
    ds = ds.chunk(time=1)
    for var in ds.data_vars:
        if 'chunks' in ds[var].encoding:
            del ds[var].encoding['chunks']
    ds.to_zarr(OUT_PATH, mode='a', append_dim='time', zarr_format=2)
    ds.close()

print(f"Done! Written to {OUT_PATH}")

Writing first file: s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/precip/precip.1979_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr


Appending files: 100%|██████████| 187/187 [1:37:49<00:00, 31.39s/it]

Done! Written to s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/precip_all_06.zarr
